In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# ---------------- CONFIG ----------------
GOLD_DB              = "workspace.slainte_gold"
FACT_TABLE           = f"{GOLD_DB}.fact_ticket_perf"
DIM_EMPLOYEE         = f"{GOLD_DB}.dim_employee"
DIM_TEAM             = f"{GOLD_DB}.dim_team"
DIM_PRIORITY         = f"{GOLD_DB}.dim_priority"
DIM_STATUS           = f"{GOLD_DB}.dim_status"
DIM_TT               = f"{GOLD_DB}.dim_ticket_type"
DIM_SLA              = f"{GOLD_DB}.dim_sla"
DIM_PENALTY          = f"{GOLD_DB}.dim_penalty"
ENRICHED_TABLE       = f"{GOLD_DB}.fact_ticket_perf_enriched"
VIEW_TICKET_ENRICHED = f"{GOLD_DB}.vw_ticket_perf_enriched"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- LOAD ----------------
f         = spark.table(FACT_TABLE).alias("f")
e         = spark.table(DIM_EMPLOYEE).alias("e")
t         = spark.table(DIM_TEAM).alias("t")
pr        = spark.table(DIM_PRIORITY).alias("pr")
st        = spark.table(DIM_STATUS).alias("st")
tt        = spark.table(DIM_TT).alias("tt")
s         = spark.table(DIM_SLA).alias("s")
pen_resp  = spark.table(DIM_PENALTY).filter(F.col("penalty_code") == "LATE_RESPONSE").alias("pen_resp")
pen_resol = spark.table(DIM_PENALTY).filter(F.col("penalty_code") == "LATE_RESOLUTION").alias("pen_resol")

# ---------------- JOIN ----------------
df_view = (
    f
    .join(e,
        (F.col("f.source_user_id") == F.col("e.source_user_id")) &
        (F.col("f.project")        == F.col("e.project"))        &
        (F.col("f.employee_id")    == F.col("e.employee_id")),
        "left"
    )
    .join(t,
        (F.col("f.source_user_id") == F.col("t.source_user_id")) &
        (F.col("f.project")        == F.col("t.project"))        &
        (F.col("f.team_id")        == F.col("t.team_id")),
        "left"
    )
    .join(pr,
        (F.col("f.source_user_id") == F.col("pr.source_user_id")) &
        (F.col("f.project")        == F.col("pr.project"))        &
        (F.col("f.priority_id")    == F.col("pr.priority_id")),
        "left"
    )
    .join(st,
        (F.col("f.source_user_id") == F.col("st.source_user_id")) &
        (F.col("f.project")        == F.col("st.project"))        &
        (F.col("f.status_id")      == F.col("st.status_id")),
        "left"
    )
    .join(tt,
        (F.col("f.source_user_id") == F.col("tt.source_user_id")) &
        (F.col("f.project")        == F.col("tt.project"))        &
        (F.col("f.ticket_type_id") == F.col("tt.ticket_type_id")),
        "left"
    )
    .join(s,
        (F.col("f.source_user_id") == F.col("s.source_user_id")) &
        (F.col("f.sla_id")         == F.col("s.sla_id")),
        "left"
    )
    .join(pen_resp,
        (F.col("f.source_user_id")      == F.col("pen_resp.source_user_id")) &
        (F.col("f.project")             == F.col("pen_resp.project"))        &
        (F.col("f.penalty_response_id") == F.col("pen_resp.penalty_id")),
        "left"
    )
    .join(pen_resol,
        (F.col("f.source_user_id")        == F.col("pen_resol.source_user_id")) &
        (F.col("f.project")               == F.col("pen_resol.project"))        &
        (F.col("f.penalty_resolution_id") == F.col("pen_resol.penalty_id")),
        "left"
    )
    .select(
        # ── IDENTIFIERS ──────────────────────────────────────────
        F.col("f.ticket_number"),
        F.col("f.source_user_id"),
        F.col("f.project"),

        # ── DIMENSIONS ───────────────────────────────────────────
        F.coalesce(F.col("e.employee_name"), F.lit("Unassigned")).alias("employee_name"),
        F.coalesce(F.col("t.team_name"),     F.lit("Unassigned")).alias("team_name"),
        F.col("pr.priority_code").alias("priority"),
        F.col("st.status_code").alias("status"),
        F.col("tt.ticket_type_name").alias("ticket_type"),

        # ── SLA TARGETS ──────────────────────────────────────────
        F.col("s.response_time_minutes").alias("sla_target_response_min"),
        F.col("s.resolution_time_hours").alias("sla_target_resolution_hrs"),

        # ── SLA ACTUALS ──────────────────────────────────────────
        (F.col("f.first_response_time_ms") / 60000).alias("actual_response_minutes"),
        (F.col("f.resolution_time_ms") / 3600000).alias("actual_resolution_hours"),
        F.col("f.first_response_breached"),
        F.col("f.resolution_breached"),

        # ── SLA PERFORMANCE KPIs ─────────────────────────────────
        F.when(F.col("f.first_response_breached") == True, 1).otherwise(0).alias("response_breach_flag"),
        F.when(F.col("f.resolution_breached")     == True, 1).otherwise(0).alias("resolution_breach_flag"),

        # SLA margin: how far actual was from target (negative = breached)
        (
            F.col("s.response_time_minutes") -
            (F.col("f.first_response_time_ms") / 60000)
        ).alias("response_sla_margin_minutes"),
        (
            F.col("s.resolution_time_hours") -
            (F.col("f.resolution_time_ms") / 3600000)
        ).alias("resolution_sla_margin_hours"),

        # SLA margin % vs target
        F.when(F.col("s.response_time_minutes") > 0,
            F.round(
                (F.col("s.response_time_minutes") - (F.col("f.first_response_time_ms") / 60000)) /
                F.col("s.response_time_minutes") * 100, 2
            )
        ).otherwise(F.lit(None)).alias("response_sla_margin_pct"),

        F.when(F.col("s.resolution_time_hours") > 0,
            F.round(
                (F.col("s.resolution_time_hours") - (F.col("f.resolution_time_ms") / 3600000)) /
                F.col("s.resolution_time_hours") * 100, 2
            )
        ).otherwise(F.lit(None)).alias("resolution_sla_margin_pct"),

        # Fully compliant flag
        F.when(
            (F.col("f.first_response_breached") == False) &
            (F.col("f.resolution_breached")     == False),
            1
        ).otherwise(0).alias("fully_compliant_flag"),

        # ── PENALTY & FINANCIAL KPIs ─────────────────────────────
        F.col("pen_resp.penalty_type").alias("response_penalty_type"),
        F.coalesce(F.col("f.response_penalty_amount"),   F.lit(0.0)).alias("response_penalty_amount"),
        F.coalesce(F.col("f.response_penalty_unit"),     F.lit("EUR")).alias("penalty_currency"),
        F.col("pen_resol.penalty_type").alias("resolution_penalty_type"),
        F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0)).alias("resolution_penalty_amount"),
        (
            F.coalesce(F.col("f.response_penalty_amount"),   F.lit(0.0)) +
            F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))
        ).alias("total_penalty_amount"),

        # Penalty exposure flag (any penalty applied)
        F.when(
            (F.coalesce(F.col("f.response_penalty_amount"),   F.lit(0.0)) +
             F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))) > 0,
            1
        ).otherwise(0).alias("has_penalty_flag"),

        # ── TICKET LIFECYCLE KPIs ────────────────────────────────
        F.col("f.created_at"),
        F.col("f.resolved_at"),
        F.col("f.due_date"),
        F.col("f.last_update"),
        F.col("f.ingestion_ts"),

        # Resolution duration
        F.datediff(F.col("f.resolved_at"), F.col("f.created_at")).alias("resolution_days"),

        # Ticket age from creation to last update
        F.datediff(F.col("f.last_update"), F.col("f.created_at")).alias("ticket_age_days"),

        # Is ticket resolved
        F.when(F.col("f.resolved_at").isNotNull(), 1).otherwise(0).alias("is_resolved_flag"),

        # Overdue: unresolved and past due date
        F.when(
            (F.col("f.resolved_at").isNull()) &
            (F.col("f.due_date").isNotNull()) &
            (F.col("f.due_date") < F.current_date()),
            1
        ).otherwise(0).alias("overdue_flag"),

        # Days until or since due date (negative = past due)
        F.datediff(F.col("f.due_date"), F.current_date()).alias("days_to_due_date"),

        # Speed tier based on actual resolution time
        F.when(F.col("f.resolution_time_ms") / 3600000 <= 1,   F.lit("< 1h"))
         .when(F.col("f.resolution_time_ms") / 3600000 <= 4,   F.lit("1–4h"))
         .when(F.col("f.resolution_time_ms") / 3600000 <= 24,  F.lit("4–24h"))
         .when(F.col("f.resolution_time_ms") / 3600000 <= 72,  F.lit("1–3 days"))
         .otherwise(F.lit("> 3 days")).alias("resolution_speed_tier"),

        # Created time buckets for trend analysis
        F.date_trunc("day",   F.col("f.created_at")).alias("created_date"),
        F.date_trunc("week",  F.col("f.created_at")).alias("created_week"),
        F.date_trunc("month", F.col("f.created_at")).alias("created_month"),

        # ── TEAM & AGENT PERFORMANCE KPIs ────────────────────────
        # Agent breach score (0 = clean, 1 = response breach, 2 = resolution breach, 3 = both)
        (
            F.when(F.col("f.first_response_breached") == True, 1).otherwise(0) +
            F.when(F.col("f.resolution_breached")     == True, 1).otherwise(0)
        ).alias("agent_breach_score"),

        # Penalty responsibility (for agent/team attribution)
        F.when(
            (F.coalesce(F.col("f.response_penalty_amount"),   F.lit(0.0)) +
             F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))) > 0,
            F.coalesce(F.col("e.employee_name"), F.lit("Unassigned"))
        ).otherwise(F.lit(None)).alias("penalized_agent"),

        F.when(
            (F.coalesce(F.col("f.response_penalty_amount"),   F.lit(0.0)) +
             F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))) > 0,
            F.coalesce(F.col("t.team_name"), F.lit("Unassigned"))
        ).otherwise(F.lit(None)).alias("penalized_team")
    )
    # ── DEDUP ────────────────────────────────────────────────────
    .withColumn(
        "_rn",
        F.row_number().over(
            Window.partitionBy("ticket_number", "source_user_id", "project")
            .orderBy(F.col("last_update").desc_nulls_last())
        )
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# ---------------- WRITE & CREATE VIEW ----------------
df_view.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(ENRICHED_TABLE)

spark.sql(f"""
    CREATE OR REPLACE VIEW {VIEW_TICKET_ENRICHED} AS
    SELECT * FROM {ENRICHED_TABLE}
""")

print(f"✅ View created successfully: {VIEW_TICKET_ENRICHED}")
display(spark.table(VIEW_TICKET_ENRICHED))

In [0]:
from pyspark.sql import functions as F

from pyspark.sql import Window

# ---------------- CONFIG ----------------

GOLD_DB              = "workspace.slainte_gold"

FACT_TABLE           = f"{GOLD_DB}.fact_ticket_perf"

DIM_EMPLOYEE         = f"{GOLD_DB}.dim_employee"

DIM_TEAM             = f"{GOLD_DB}.dim_team"

DIM_PRIORITY         = f"{GOLD_DB}.dim_priority"

DIM_STATUS           = f"{GOLD_DB}.dim_status"

DIM_TT               = f"{GOLD_DB}.dim_ticket_type"

DIM_SLA              = f"{GOLD_DB}.dim_sla"

DIM_PENALTY          = f"{GOLD_DB}.dim_penalty"

ENRICHED_TABLE       = f"{GOLD_DB}.fact_ticket_perf_enriched"

VIEW_TICKET_ENRICHED = f"{GOLD_DB}.vw_ticket_perf_enriched"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- LOAD ----------------

f         = spark.table(FACT_TABLE).alias("f")

e         = spark.table(DIM_EMPLOYEE).alias("e")

t         = spark.table(DIM_TEAM).alias("t")

pr        = spark.table(DIM_PRIORITY).alias("pr")

st        = spark.table(DIM_STATUS).alias("st")

tt        = spark.table(DIM_TT).alias("tt")

s         = spark.table(DIM_SLA).alias("s")

pen_resp  = spark.table(DIM_PENALTY).filter(F.col("penalty_code") == "LATE_RESPONSE").alias("pen_resp")

pen_resol = spark.table(DIM_PENALTY).filter(F.col("penalty_code") == "LATE_RESOLUTION").alias("pen_resol")

# ---------------- JOIN ----------------

df_view = (

    f

    .join(

        e,

        (F.col("f.source_user_id") == F.col("e.source_user_id")) &

        (F.col("f.project") == F.col("e.project")) &

        (F.col("f.employee_id") == F.col("e.employee_id")),

        "left"

    )

    .join(

        t,

        (F.col("f.source_user_id") == F.col("t.source_user_id")) &

        (F.col("f.project") == F.col("t.project")) &

        (F.col("f.team_id") == F.col("t.team_id")),

        "left"

    )

    .join(

        pr,

        (F.col("f.source_user_id") == F.col("pr.source_user_id")) &

        (F.col("f.project") == F.col("pr.project")) &

        (F.col("f.priority_id") == F.col("pr.priority_id")),

        "left"

    )

    .join(

        st,

        (F.col("f.source_user_id") == F.col("st.source_user_id")) &

        (F.col("f.project") == F.col("st.project")) &

        (F.col("f.status_id") == F.col("st.status_id")),

        "left"

    )

    .join(

        tt,

        (F.col("f.source_user_id") == F.col("tt.source_user_id")) &

        (F.col("f.project") == F.col("tt.project")) &

        (F.col("f.ticket_type_id") == F.col("tt.ticket_type_id")),

        "left"

    )

    .join(

        s,

        (F.col("f.source_user_id") == F.col("s.source_user_id")) &

        (F.col("f.sla_id") == F.col("s.sla_id")),

        "left"

    )

    .join(

        pen_resp,

        (F.col("f.source_user_id") == F.col("pen_resp.source_user_id")) &

        (F.col("f.project") == F.col("pen_resp.project")) &

        (F.col("f.penalty_response_id") == F.col("pen_resp.penalty_id")),

        "left"

    )

    .join(

        pen_resol,

        (F.col("f.source_user_id") == F.col("pen_resol.source_user_id")) &

        (F.col("f.project") == F.col("pen_resol.project")) &

        (F.col("f.penalty_resolution_id") == F.col("pen_resol.penalty_id")),

        "left"

    )

    .select(

        # ── IDENTIFIERS ──────────────────────────────────────────

        F.col("f.ticket_number"),

        F.col("f.source_user_id"),

        F.col("f.project"),

        # ── DIMENSIONS ───────────────────────────────────────────

        F.coalesce(F.col("e.employee_name"), F.lit("Unassigned")).alias("employee_name"),

        F.coalesce(F.col("t.team_name"), F.lit("Unassigned")).alias("team_name"),

        F.col("pr.priority_code").alias("priority"),

        F.col("st.status_code").alias("status"),

        F.col("tt.ticket_type_name").alias("ticket_type"),

        # ── SLA TARGETS ──────────────────────────────────────────

        F.col("s.response_time_minutes").alias("sla_target_response_min"),

        F.col("s.resolution_time_hours").alias("sla_target_resolution_hrs"),

        # ── SLA ACTUALS ──────────────────────────────────────────

        (F.col("f.first_response_time_ms") / 60000).alias("actual_response_minutes"),

        (F.col("f.resolution_time_ms") / 3600000).alias("actual_resolution_hours"),

        F.col("f.first_response_breached"),

        F.col("f.resolution_breached"),

        # ── SLA PERFORMANCE KPIs ─────────────────────────────────

        F.when(F.col("f.first_response_breached") == True, 1).otherwise(0).alias("response_breach_flag"),

        F.when(F.col("f.resolution_breached") == True, 1).otherwise(0).alias("resolution_breach_flag"),

        # SLA margin: how far actual was from target (negative = breached)

        (

            F.col("s.response_time_minutes") -

            (F.col("f.first_response_time_ms") / 60000)

        ).alias("response_sla_margin_minutes"),

        (

            F.col("s.resolution_time_hours") -

            (F.col("f.resolution_time_ms") / 3600000)

        ).alias("resolution_sla_margin_hours"),

        # SLA margin % vs target

        F.when(F.col("s.response_time_minutes") > 0,

            F.round(

                (F.col("s.response_time_minutes") - (F.col("f.first_response_time_ms") / 60000)) /

                F.col("s.response_time_minutes") * 100, 2

            )

        ).otherwise(F.lit(None)).alias("response_sla_margin_pct"),

        F.when(F.col("s.resolution_time_hours") > 0,

            F.round(

                (F.col("s.resolution_time_hours") - (F.col("f.resolution_time_ms") / 3600000)) /

                F.col("s.resolution_time_hours") * 100, 2

            )

        ).otherwise(F.lit(None)).alias("resolution_sla_margin_pct"),

        # Fully compliant flag

        F.when(

            (F.col("f.first_response_breached") == False) &

            (F.col("f.resolution_breached") == False),

            1

        ).otherwise(0).alias("fully_compliant_flag"),

        # ── PENALTY & FINANCIAL KPIs ─────────────────────────────

        F.col("pen_resp.penalty_type").alias("response_penalty_type"),

        F.coalesce(F.col("f.response_penalty_amount"), F.lit(0.0)).alias("response_penalty_amount"),

        F.coalesce(F.col("f.response_penalty_unit"), F.lit("EUR")).alias("penalty_currency"),

        F.col("pen_resol.penalty_type").alias("resolution_penalty_type"),

        F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0)).alias("resolution_penalty_amount"),

        (

            F.coalesce(F.col("f.response_penalty_amount"), F.lit(0.0)) +

            F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))

        ).alias("total_penalty_amount"),

        # Penalty exposure flag (any penalty applied)

        F.when(

            (F.coalesce(F.col("f.response_penalty_amount"), F.lit(0.0)) +

             F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))) > 0,

            1

        ).otherwise(0).alias("has_penalty_flag"),

        # ── TICKET LIFECYCLE KPIs ────────────────────────────────

        F.col("f.created_at"),

        F.col("f.resolved_at"),

        F.col("f.due_date"),

        F.col("f.last_update"),

        F.col("f.ingestion_ts"),

        # Resolution duration

        F.datediff(F.col("f.resolved_at"), F.col("f.created_at")).alias("resolution_days"),

        # Ticket age from creation to last update

        F.datediff(F.col("f.last_update"), F.col("f.created_at")).alias("ticket_age_days"),

        # Is ticket resolved

        F.when(F.col("f.resolved_at").isNotNull(), 1).otherwise(0).alias("is_resolved_flag"),

        # Overdue: unresolved and past due date

        F.when(

            (F.col("f.resolved_at").isNull()) &

            (F.col("f.due_date").isNotNull()) &

            (F.col("f.due_date") < F.current_date()),

            1

        ).otherwise(0).alias("overdue_flag"),

        # Days until or since due date (negative = past due)

        F.datediff(F.col("f.due_date"), F.current_date()).alias("days_to_due_date"),

        # Speed tier based on actual resolution time

        F.when(F.col("f.resolution_time_ms") / 3600000 <= 1, F.lit("< 1h"))

         .when(F.col("f.resolution_time_ms") / 3600000 <= 4, F.lit("1–4h"))

         .when(F.col("f.resolution_time_ms") / 3600000 <= 24, F.lit("4–24h"))

         .when(F.col("f.resolution_time_ms") / 3600000 <= 72, F.lit("1–3 days"))

         .otherwise(F.lit("> 3 days")).alias("resolution_speed_tier"),

        # Created time buckets for trend analysis

        F.date_trunc("day", F.col("f.created_at")).alias("created_date"),

        F.date_trunc("week", F.col("f.created_at")).alias("created_week"),

        F.date_trunc("month", F.col("f.created_at")).alias("created_month"),

        # ── TEAM & AGENT PERFORMANCE KPIs ────────────────────────

        (

            F.when(F.col("f.first_response_breached") == True, 1).otherwise(0) +

            F.when(F.col("f.resolution_breached") == True, 1).otherwise(0)

        ).alias("agent_breach_score"),

        # Penalty responsibility (for agent/team attribution)

        F.when(

            (F.coalesce(F.col("f.response_penalty_amount"), F.lit(0.0)) +

             F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))) > 0,

            F.coalesce(F.col("e.employee_name"), F.lit("Unassigned"))

        ).otherwise(F.lit(None)).alias("penalized_agent"),

        F.when(

            (F.coalesce(F.col("f.response_penalty_amount"), F.lit(0.0)) +

             F.coalesce(F.col("f.resolution_penalty_amount"), F.lit(0.0))) > 0,

            F.coalesce(F.col("t.team_name"), F.lit("Unassigned"))

        ).otherwise(F.lit(None)).alias("penalized_team")

    )

    # ── NEW PER-TICKET KPI COLUMNS ───────────────────────────────

    .withColumn(

        "total_breach_per_ticket",

        F.col("response_breach_flag") + F.col("resolution_breach_flag")

    )

    .withColumn(

        "sla_compliance_pct_per_ticket",

        F.round(

            ((F.lit(2) - F.col("total_breach_per_ticket")) / F.lit(2)) * 100,

            2

        )

    )

    # ── DEDUP ────────────────────────────────────────────────────

    .withColumn(

        "_rn",

        F.row_number().over(

            Window.partitionBy("ticket_number", "source_user_id", "project")

            .orderBy(F.col("last_update").desc_nulls_last())

        )

    )

    .filter(F.col("_rn") == 1)

    .drop("_rn")

)

# ---------------- WRITE & CREATE VIEW ----------------

df_view.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(ENRICHED_TABLE)

spark.sql(f"""

    CREATE OR REPLACE VIEW {VIEW_TICKET_ENRICHED} AS

    SELECT * FROM {ENRICHED_TABLE}

""")

print(f"✅ View created successfully: {VIEW_TICKET_ENRICHED}")

display(spark.table(VIEW_TICKET_ENRICHED))
 